### **Setup: Imports und Initialisierung**

In [2]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from quinex_application.quinex_client import QuinexClient
from quinex_application.visualisation_metadata import VisualisationMetadata
from quinex_application.mongos_store import MongoStore

In [2]:
quinex = QuinexClient()
visualisation = VisualisationMetadata("../data/paper/paper_dataset.parquet")
mongodb = MongoStore(quinex, database_name="quinex_db_query4", collection_name="papers_2011")

### **Create Bulk Analysis**

In [3]:
name = "quinex_application_query4"
response = quinex.intialize_analysis(name)
print(response)

{'detail': 'Analysis with name quinex_application_query4 already exists. Please choose another name.'}


### **Import and Sort Paper by Year**

In [4]:
visualisation.load_df( query='Query: TITLE-ABS-KEY ( ( "energy system*" ) AND ( simul* OR model* OR "plan" OR "plans" OR "planning" OR optimi* OR analy* OR assess* OR evaluat* ) ) AND ( LIMIT-TO ( OA , "all" ) ) AND ( LIMIT-TO ( LANGUAGE , "English" ) )', date="14.05.2025")
df_doi = visualisation.df

In [5]:
# Sort by year
dfs_by_year = {
    year: group_df
    for year, group_df in df_doi.groupby("Year")
}
df_2011 = dfs_by_year[2011]
print(len(df_2011))

201


### **Add Papers via DOI**

In [6]:
mongodb.insert_paper(name, df_2011)

Verarbeitet: 201/201 | ✅ Neu: 111 | 🔁 Vorhanden: 0 | ❌ Fehlgeschlagen: 90
📚 Total DOIs currently stored: 111


#### **Finish remaining Papers**

In [ ]:
# Hole die DOIs (_id) aus beiden Collections als Sets
papers_2013_ids = {doc["_id"] for doc in mongodb.get_collection("papers_2013")}
failed_papers_ids = {doc["_id"] for doc in mongodb.get_collection("failed_papers")}

# Vereine beide Sets (alle DOIs, die ausgeschlossen werden sollen)
excluded_ids = papers_2013_ids.union(failed_papers_ids)

# Filtere das DataFrame: nur Zeilen mit DOIs, die NICHT in excluded_ids enthalten sind
filtered_df = df_2013[~df_2013["DOI"].isin(excluded_ids)]
print(len(filtered_df))

In [ ]:
mongodb.insert_paper(name, filtered_df)

### **Successfully Added Papers**

In [20]:
dois_df = mongodb.get_papers_from_collection(df_doi, "papers_2012")
print(len(dois_df))

162


### **Perform Bulk Analysis**

In [ ]:
# Vor Start:
# 9283806    normal quinex_b j-goepfe  R 15-02:07:14      1 cn31
# 9083722    normal quinex_b j-goepfe  R 15-02:06:41      1 cn26

# Nach Start:
#    9283806    normal quinex_b j-goepfe  R 25-02:47:23      1 cn31
#    9083722    normal quinex_b j-goepfe  R 7-22:06:11      1 cn26
#   11875617    normal quinex_b j-goepfe  R      45:10      1 cn22
#   11875618    normal quinex_b j-goepfe  R      45:08      1 cn22
#   11875619    normal quinex_b j-goepfe  R      45:08      1 cn22
#   11875620    normal quinex_b j-goepfe  R      45:08      1 cn22
#   11875621    normal quinex_b j-goepfe  R      45:08      1 cn22
#   11875622    normal quinex_b j-goepfe  R      45:08      1 cn22
#   11875623    normal quinex_b j-goepfe  R      45:08      1 cn22
#   11875624    normal quinex_b j-goepfe  R      45:08      1 cn22


In [ ]:
dois = [doi for doi in dois_df["DOI"]]
mongodb.reset_quinex_result(dois)
mongodb.process_paper(name, dois, number_of_gpus=8)

🔄 Reset: 0 of 162 requested DOIs.


### **Add Bulk Analysis to MongoDB**

In [ ]:
dois = [doi for doi in dois_df["DOI"]]
mongodb.reset_quinex_result(dois)
mongodb.add_analysis(name, dois)

In [ ]:
analysis = mongodb.get_collection("papers_2015")

### **Update Paper Parquet**

In [3]:
from quinex_application.quinex_postprocessing import QuinexPostprocessing
qpost = QuinexPostprocessing()
df = qpost.load_df("../data/paper/paper_dataset.parquet")

📂 Loaded DataFrame from '../data/paper/paper_dataset.parquet' with 184128 rows


In [ ]:
display(df.head())

In [ ]:
df = mongodb.update_paper_parquet("papers_2012", "../data/paper/paper_dataset.parquet", "29.07.2025", "17.04.2025")

In [4]:
print("-------------------QUINEX Performed-------------------------------")
print(len(df[(df["Quinex"] == True)]))
print("-------------------QUINEX Tried-----------------------------------")
print(len(df[(df["Download_Status"] == "Succeeded")]) + len(df[(df["Download_Status"] == "Failed")]))
print("-------------------Overall Papers-------------------------------")
print(len(df[(df['Query: TITLE-ABS-KEY ( ( "energy system*" ) AND ( simul* OR model* OR "plan" OR "plans" OR "planning" OR optimi* OR analy* OR assess* OR evaluat* ) ) AND ( LIMIT-TO ( OA , "all" ) ) AND ( LIMIT-TO ( LANGUAGE , "English" ) )_14.05.2025'] == True)]))
print("-------------------Additional WOS Papers-------------------------------")
print(len(df[(df['Query: AB=("energy system*" AND (simul* OR model* OR plan OR plans OR planning OR optimi* OR analy* OR assess* OR evaluat*))_10.06.2025'] == True) & (df['Query: TITLE-ABS-KEY ( ( "energy system*" ) AND ( simul* OR model* OR "plan" OR "plans" OR "planning" OR optimi* OR analy* OR assess* OR evaluat* ) ) AND ( LIMIT-TO ( OA , "all" ) ) AND ( LIMIT-TO ( LANGUAGE , "English" ) )_14.05.2025'] == False)]))

-------------------QUINEX Performed-------------------------------
13840
-------------------QUINEX Tried-----------------------------------
24253
-------------------Overall Papers-------------------------------
25384
-------------------Additional WOS Papers-------------------------------
864
